# Governance-hierarchy graph

A small graph of building rules across three levels of government (federal, state,
municipal), built from the output of a local language model.

**How the graph was made**

1. I wrote 20 short test sentences in German, modelled on patterns found in Stuttgart
   development plans. They are constructed test cases, not quotations from the plans
   or from the law.
2. A local model (qwen2.5:7b through Ollama) extracted provisions and relationships
   from each sentence. See `extract_relationships_local.py`.
3. First pass: 16 of 20 correct. Three misses were override relationships that the
   model invented because of override-sounding words. One was a real relationship it
   missed because it merged two paragraph numbers into one entity.
4. I added targeted examples to the prompt and re-ran the 4 failed sentences: 4 of 4
   then correct.
5. I cleaned the output by hand into `data/nodes.csv` and `data/edges.csv`.

Read the README for the limits of this result before drawing conclusions from it.


## 1. Imports

In [1]:
import pandas as pd
import networkx as nx
import plotly.graph_objects as go


## 2. Load the graph

`nodes.csv` and `edges.csv` hold the provisions and relationships after the prompt
fix. Each row notes which test sentence it came from and whether it was correct on
the first pass.


In [2]:
nodes = pd.read_csv("data/nodes.csv")
edges = pd.read_csv("data/edges.csv")

print(f"{len(nodes)} nodes, {len(edges)} edges — from 20 test sentences")
display(nodes[["node_id", "tier", "topic", "notes"]])


21 nodes, 5 edges — from 20 test sentences


,node_id,tier,topic,notes
0,BauNVO_17_GFZ,federal,GFZ,Federal baseline GFZ 1.2 for reine Wohngebiete
1,Zuffenhausen_WB,municipal,GFZ,Overrides federal GFZ default with 1.6 -- expl...
2,Stgt_322_1_Abstandsflaeche,municipal,setback,Overrides LBO BW default setback with 2.5m
3,LBO_BW_Abstandsflaeche_default,state,setback,Baseline setback rule being overridden
4,Muenster41_SG_GRZ,municipal,GRZ,Overrides regelhafte GRZ-Obergrenze with reduc...
5,GRZ_regelhafte_Obergrenze,state,GRZ,Baseline GRZ ceiling being overridden
6,Weilimdorf_222_SO,municipal,GRZ,Overrides Regelbebauung with reduced GRZ 0.25
7,GRZ_Regelbebauung,state,GRZ,Baseline being overridden
8,LBO_BW_Paragraph_5,state,fire_safety,"Load-bearing walls, Gebaeudeklasse 5"
9,LBO_BW_Paragraph_10,state,fire_safety,"Supplementary fire-safety measures, same Gebae..."


## 3. Build the graph

In [3]:
G = nx.DiGraph()

for _, row in nodes.iterrows():
    G.add_node(row["node_id"], **row.drop("node_id").dropna().to_dict())

for _, row in edges.iterrows():
    G.add_edge(row["source"], row["target"],
               relationship=row["relationship"], confidence=row["confidence"])

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Isolated nodes (no relationship found, correctly): "
      f"{sum(1 for n in G.nodes() if G.degree(n) == 0)}")


Graph built: 21 nodes, 5 edges
Isolated nodes (no relationship found, correctly): 11


## 4. Visualize

Connected nodes (real `overrides` / `same_provision_group` relationships) cluster
together; isolated nodes are the ones the pipeline correctly recognized as
having NO relationship — including several deliberately tricky cases (same zone
label, different plan; same plan, different sub-zone) that a naive
lexical-matching approach would likely have linked incorrectly.

In [4]:
# v3 layout: same tier-hierarchy idea as before, but with real spacing math
# instead of guessed constants -- horizontal spacing is computed FROM the
# longest label in each row, not a fixed number, so labels physically cannot
# collide regardless of how many nodes land in one tier.

tier_y = {"federal": 3, "state": 2, "municipal": 1}
tier_colors = {"federal": "#D4A017", "state": "#E67E22", "municipal": "#8E44AD"}

tiers = {}
for n in G.nodes():
    tiers.setdefault(G.nodes[n].get("tier", "municipal"), []).append(n)

# Estimate required horizontal spacing per tier from label length (rough:
# ~0.09 plot-units per character at this font size, plus a fixed margin).
SPACING_PER_CHAR = 0.16  # includes real margin, not just exact label width
MIN_SPACING = 3.2

pos = {}
for tier, nodelist in tiers.items():
    connected = sorted([n for n in nodelist if G.degree(n) > 0])
    isolated = sorted([n for n in nodelist if G.degree(n) == 0])
    ordered = connected + isolated  # connected nodes on the left, isolated on the right
    max_label_len = max(len(n.replace("_"," ")) for n in ordered) if ordered else 10
    spacing = max(MIN_SPACING, max_label_len * SPACING_PER_CHAR)
    n_total = len(ordered)
    for i, n in enumerate(ordered):
        x = i * spacing
        pos[n] = (x, tier_y.get(tier, 1))

# Center each tier row independently so rows don't drift relative to each other
for tier, nodelist in tiers.items():
    xs = [pos[n][0] for n in nodelist]
    offset = (max(xs) + min(xs)) / 2 if xs else 0
    for n in nodelist:
        pos[n] = (pos[n][0] - offset, pos[n][1])

# --- Edges: color by relationship type, directional arrowheads ---
rel_colors = {"overrides": "#2E8B57", "same_provision_group": "#4682B4"}
edge_traces = []
annotations = []
for u, v, d in G.edges(data=True):
    x0, y0 = pos[u]; x1, y1 = pos[v]
    rel = d["relationship"]
    color = rel_colors.get(rel, "grey")
    edge_traces.append(go.Scatter(
        x=[x0, x1, None], y=[y0, y1, None], mode="lines",
        line=dict(width=3, color=color),
        hoverinfo="text", text=f"{u} --[{rel}]--> {v}", showlegend=False
    ))
    annotations.append(dict(
        ax=x0, ay=y0, x=x0 + 0.85*(x1-x0), y=y0 + 0.85*(y1-y0),
        xref="x", yref="y", axref="x", ayref="y",
        showarrow=True, arrowhead=3, arrowsize=1.5, arrowwidth=2.2, arrowcolor=color
    ))

# --- Nodes: connected = vivid + labeled above; isolated = muted grey +
#     labeled below, smaller, so they visually recede as "correctly rejected
#     background" rather than competing for attention with the real signal ---
node_x, node_y, node_color, node_size, node_line, label_text, label_pos, label_font = [],[],[],[],[],[],[],[]
hover = []

for n in G.nodes():
    x, y = pos[n]
    connected = G.degree(n) > 0
    node_x.append(x); node_y.append(y)
    if connected:
        node_color.append(tier_colors.get(G.nodes[n].get("tier"), "grey"))
        node_size.append(30)
        node_line.append("black")
        label_pos.append("top center")
        label_font.append(11)
    else:
        node_color.append("#D9D9D9")
        node_size.append(14)
        node_line.append("#AAAAAA")
        label_pos.append("bottom center")
        label_font.append(8)
    label_text.append(n.replace("_", " "))
    hover.append(f"<b>{n}</b><br>tier: {G.nodes[n].get('tier','')}<br>{G.nodes[n].get('notes','')}")

node_trace = go.Scatter(
    x=node_x, y=node_y, mode="markers+text",
    text=label_text, hovertext=hover, hoverinfo="text",
    marker=dict(size=node_size, color=node_color,
                line=dict(width=1.3, color=node_line)),
    showlegend=False
)
# textposition/textfont must be set per-point via array (Plotly supports this)
node_trace.textposition = label_pos
node_trace.textfont = dict(size=label_font, color="black")

legend_traces = [
    go.Scatter(x=[None], y=[None], mode="lines", line=dict(width=3, color="#2E8B57"), name="overrides"),
    go.Scatter(x=[None], y=[None], mode="lines", line=dict(width=3, color="#4682B4"), name="same_provision_group"),
    go.Scatter(x=[None], y=[None], mode="markers", marker=dict(size=14, color="#D4A017"), name="federal tier"),
    go.Scatter(x=[None], y=[None], mode="markers", marker=dict(size=14, color="#E67E22"), name="state tier"),
    go.Scatter(x=[None], y=[None], mode="markers", marker=dict(size=14, color="#8E44AD"), name="municipal tier"),
    go.Scatter(x=[None], y=[None], mode="markers", marker=dict(size=10, color="#D9D9D9"), name="isolated (correctly no relationship)"),
]

# Subtle background bands per tier row -- reinforces the hierarchy visually
tier_bands = []
all_x = [p[0] for p in pos.values()]
xmin, xmax = min(all_x) - 2, max(all_x) + 2
band_colors = {"federal": "rgba(212,160,23,0.06)", "state": "rgba(230,126,34,0.06)", "municipal": "rgba(142,68,173,0.06)"}
for tier, y in tier_y.items():
    tier_bands.append(dict(type="rect", x0=xmin, x1=xmax, y0=y-0.4, y1=y+0.4,
                            fillcolor=band_colors.get(tier, "rgba(0,0,0,0.03)"),
                            line=dict(width=0), layer="below"))

fig = go.Figure(data=edge_traces + [node_trace] + legend_traces)
fig.update_layout(
    title="Governance-Hierarchy Graph — spacing computed from label width, not guessed<br>"
          "<sub>Vivid = real relationship. Grey/small = correctly rejected. Arrows show override direction.</sub>",
    annotations=annotations,
    shapes=tier_bands,
    showlegend=True,
    legend=dict(x=1.02, y=1, bgcolor="rgba(255,255,255,0.9)", bordercolor="lightgrey", borderwidth=1),
    height=750, width=1500,
    xaxis=dict(visible=False, range=[xmin-1, xmax+1]),
    yaxis=dict(visible=False, range=[0.3, 3.7]),
    plot_bgcolor="white", margin=dict(l=20, r=180, t=80, b=20),
    updatemenus=[dict(
        type="buttons", direction="left", x=0.0, y=1.1,
        buttons=[
            dict(label="Show all nodes", method="restyle", args=[{"marker.opacity": 1}]),
            dict(label="Highlight connected only", method="restyle",
                 args=[{"marker.opacity": [1 if G.degree(n) > 0 else 0.2 for n in G.nodes()]}]),
        ]
    )]
)
fig.show()


## 5. Graph density and centrality on real extracted data

Density and centrality on the small extracted graph. With 5 edges among 21 nodes
these numbers are close to zero and mainly confirm that most provisions stand alone.

In [5]:
undirected = G.to_undirected()
density = nx.density(G)
closeness = nx.closeness_centrality(undirected)
betweenness = nx.betweenness_centrality(undirected)

summary = pd.DataFrame({
    "node_id": list(G.nodes()),
    "tier": [G.nodes[n].get("tier","") for n in G.nodes()],
    "degree": [G.degree(n) for n in G.nodes()],
    "closeness": [round(closeness[n], 3) for n in G.nodes()],
    "betweenness": [round(betweenness[n], 3) for n in G.nodes()],
}).sort_values("degree", ascending=False)

print(f"Graph density: {density:.3f}  (mostly isolated nodes = correctly-rejected false relationships)")
summary.head(10)


Graph density: 0.012  (mostly isolated nodes = correctly-rejected false relationships)


,node_id,tier,degree,closeness,betweenness
0,BauNVO_17_GFZ,federal,1,0.05,0.0
1,Zuffenhausen_WB,municipal,1,0.05,0.0
2,Stgt_322_1_Abstandsflaeche,municipal,1,0.05,0.0
3,LBO_BW_Abstandsflaeche_default,state,1,0.05,0.0
4,Muenster41_SG_GRZ,municipal,1,0.05,0.0
5,GRZ_regelhafte_Obergrenze,state,1,0.05,0.0
6,Weilimdorf_222_SO,municipal,1,0.05,0.0
7,GRZ_Regelbebauung,state,1,0.05,0.0
8,LBO_BW_Paragraph_5,state,1,0.05,0.0
9,LBO_BW_Paragraph_10,state,1,0.05,0.0


## 6. What this shows, and what is still open

**Shown, at small scale:**
- A local 7B model can extract provisions and override relationships from short,
  clear sentences.
- Its errors were not random. It reacted to override-sounding words without checking
  that a baseline rule was named.
- Targeted examples in the prompt corrected the 4 failed sentences.

**Not shown:**
- Whether the fix generalises. The added examples are close to the failed sentences,
  so the retest does not answer this.
- Whether quality holds on real plan text and at the scale of the full corpus
  (about 12,700 chunks).
- How the graph would handle a disagreeing source such as the vision or cadastral
  layer. It represents text only.
